 # 微分方程建模实例

### 3. 美国人口的预报模型

(2). 参数估计

把文件Pdata8_10.txt中的第一个数据作为初始条件，利用余下的数据拟合式子中的参数$x_m$和$r$


1）非线性最小二乘法

In [6]:
import numpy as np
from scipy.optimize import curve_fit

a = []
b = []
"""
用 with 语句打开数据文件并把它绑定到对象f. 不必操心在操作完资源后去关闭数据文件，因为 with 语句的上下文管理器会帮助处理．这在操作资
源型文件时非常方便，因为它能确保在代码执行完毕后资源会被释放掉（比如关闭文件）
"""
with open("Pdata8_10_1.txt") as f:  # 打开并绑定对象f
    s = f.read().splitlines()  # 返回每一行的数据
    
for i in range(0, len(s), 2):  # 读入奇数行数据
    d1 = s[i].split("\t")
    for j in range(len(d1)):
        if d1 [j] != "":
            a.append(eval(d1[j]))  # 把非空的字符串转换为年代数据
for i in range(1, len(s), 2):  # 读入偶数行数据
    d2 = s[i].split("\t")
    for j in range(len(d2)):
        if d2[j] != "":
            b.append(eval(d2[j]))  # 把非空的字符串转换为人口数据
            
c = np.vstack((a, b))  # 构造两行的数组
np.savetxt("cleaned_data.txt", c)  # 把数据保存起来供下面使用

x0 = 3.9  # txt文件中的第一个人口数量为3.9
t0 = 1790 # txt文件中的第一个时间为1790
x = lambda t, r, xm: xm / (1 + (xm / 3.9 - 1) * np.exp(-r * (t - t0)))  # 把解用函数表示
bd = ((0, 200), (0.1, 1000))  # 约束两个参数的下界和上界
popt, pcov = curve_fit(x, a[1:], b[1:], bounds=bd)
print(popt)
print(f"2010年的预测值为:{x(2010, *popt)}")

[2.73527906e-02 3.42441913e+02]
2010年的预测值为:282.679783219587


2）线性最小二乘法

为了利用渐大的线性最小二乘法估计这个模型的参数 $r$ 和 $x_m$, 把 $Logistic$ 方程表示为
$$
\frac{1}{x}·\frac{dx}{dt} \, = \, r - sx, \quad s = \frac{r}{x_m}
$$
计 1790，1800，···，2000 年分别用 k = 1，2，···，22 表示，利用向前差分，得到差分方程
$$
\frac{1}{x(k)}·\frac{x(k+1) - x(k)}{\Delta t} = r - sx(k), \quad k = 1, 2, \cdots ,21
$$
其中，步长 $\Delta t = 10$，下面先你和其中的参数 $r$ 和 $s$

In [10]:
import numpy as np

d = np.loadtxt("cleaned_data.txt")  # 加载文件中的数据
t0 = d[0]  # year
x0 = d[1]  # population
# np.diff(x0) 得到相邻人口差值，长度 N-1，前一项减后一项
b = np.diff(x0) / 10 / x0[:-1]  # 构造线性方程组的常数项列
# 全一对应截距，-x0[:-1](n-1个)对应-r/xm(斜率)，转置构成系数矩阵形式(n-1, 2)
a = np.vstack((np.ones(len(x0) - 1), -x0[:-1])).T  # 构造线性方程组系数矩阵
# pinv专门求解线性最小二乘法（不返回误差），矩阵 a 的 Moore-Penrose 伪逆（a不是方阵也可以计算）
# @ 是矩阵乘法，等价于 np.dot(pinv(a), b)
rs = np.linalg.pinv(a) @ b  # 求未知数r和xm，将系数矩阵的逆左乘到常数列向量
r = rs[0]  # 第一行为r
xm = r / rs[1]  # 第二行为xm
print(f"人口增长率r和人口最大值xm的拟合值分别为:{np.round([r, xm], 4)}")
xhat = xm / (1 + (xm / 3.9 - 1) * np.exp(-r * (2010 - 1790)))  # 求预测值
print(f"2010年的预测值为:{round(xhat, 4)}")

人口增长率r和人口最大值xm的拟合值分别为:[3.25000e-02 2.94386e+02]
2010年的预测值为:277.9634


In [13]:
import numpy as np
s0 = 155.0
i0 = 1.0
s_inf = 60.0
sigma = (np.log(s0) - np.log(s_inf)) / (s0 + i0 - s_inf)
print(f"sigma:\n{sigma}")
S = np.array([155, 153, 139, 101])
I = (s0 + i0) - S + 1 * np.log(S / s0) / sigma
print(f"所求解为:\n{I}")

sigma:
0.009886255778095274
所求解为:
[ 1.          1.6863383   5.97953014 11.67676321]


所求解过与实际数据相差较大，说明所建模型不够理想，要继续完善